In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "your_openai_api_key"


In [2]:
!pip install -q langchain-openai langchain-core requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 4.6 MB/s eta 0:00:00


In [3]:
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [4]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [5]:
print(multiply.invoke({'a':3, 'b':4}))

12


In [6]:
# tool binding

In [7]:
llm=ChatOpenAI()

In [8]:
llm_with_tools=llm.bind_tools([multiply])

In [9]:
llm_with_tools.invoke('Hi how are you')

AIMessage(content="Hello! I'm here and ready to assist you. How can I help you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 58, 'total_tokens': 77, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DK1v3hd9jzTGCv9fn25Chqe6xjcmE', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cf6c0-0fdc-7023-8723-6210017f4b2c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 58, 'output_tokens': 19, 'total_tokens': 77, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [43]:
query=HumanMessage('can you multiply 45 and 7')

In [44]:
messages=[query]

In [45]:
messages

[HumanMessage(content='can you multiply 45 and 7', additional_kwargs={}, response_metadata={})]

In [46]:
# tool calling

In [47]:
results=llm_with_tools.invoke(messages)

In [35]:
messages.append(results)

In [36]:
messages

[HumanMessage(content='can you multiply 45 and 7', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 62, 'total_tokens': 79, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DK2lIRuaGwkPtuPGVG9up7DRFJZi2', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cf6f1-7edd-7642-b259-1776fb14244e-0', tool_calls=[{'name': 'multiply', 'args': {'a': 45, 'b': 7}, 'id': 'call_2kAM3CYDHUcPcvgBApSMz12a', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 62, 'output_tokens': 17, 'total_tokens': 79, 'input_token_details': {'audio': 0, 'cache_re

In [37]:
tool_result = multiply.invoke(results.tool_calls[0])

In [38]:
tool_result

ToolMessage(content='315', name='multiply', tool_call_id='call_2kAM3CYDHUcPcvgBApSMz12a')

In [40]:
messages.append(tool_result)

In [41]:
llm_with_tools.invoke(messages).content

'The product of 45 and 7 is 315.'

In [49]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/d1289225724fdd78670a8747/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [51]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [50]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1773619201,
 'time_last_update_utc': 'Mon, 16 Mar 2026 00:00:01 +0000',
 'time_next_update_unix': 1773705601,
 'time_next_update_utc': 'Tue, 17 Mar 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 92.7043}

In [53]:
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

851.5999999999999

In [85]:
llm=ChatOpenAI()

In [86]:
llm_with_tools=llm.bind_tools([get_conversion_factor, convert])

In [87]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 usd to inr')]

In [88]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={})]

In [89]:
ai_message=llm_with_tools.invoke(messages)

In [90]:
ai_message

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 123, 'total_tokens': 175, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DK3gv5INAkSZo8dDY6AlLoZoIu4vz', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cf728-0317-7c61-98b0-1b865208a3a8-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_3aSS2P6SOVIGyXptjcvj5BZx', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': 10}, 'id': 'call_fQeM98O5QNTKvvfGxnMWSvI9', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 123, 'output_tokens':

In [91]:
messages.append(ai_message)

In [92]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_3aSS2P6SOVIGyXptjcvj5BZx',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'call_fQeM98O5QNTKvvfGxnMWSvI9',
  'type': 'tool_call'}]

In [93]:
import json
for tool_call in ai_message.tool_calls:
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    print(tool_message1.content)
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
   #execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)




{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1773619201, "time_last_update_utc": "Mon, 16 Mar 2026 00:00:01 +0000", "time_next_update_unix": 1773705601, "time_next_update_utc": "Tue, 17 Mar 2026 00:00:01 +0000", "base_code": "USD", "target_code": "INR", "conversion_rate": 92.7043}


In [94]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 123, 'total_tokens': 175, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DK3gv5INAkSZo8dDY6AlLoZoIu4vz', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cf728-0317-7c61-98b0-1b865208a3a8-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_3aSS2P6SOVIGyXptjcvj5BZx', 'type': 'tool_call'}, {'name': 'convert', 'args

In [95]:
llm_with_tools.invoke(messages).content

'The conversion factor between USD and INR is 92.7043. \n\nBased on that, 10 USD is equivalent to 927.043 INR.'